In [ ]:
"""
This jupyter notebook links entities of the RDF knowledge graphs Linked Papers With Code (LPWC) (https://linkedpaperswithcode.com) and MLSea (https://w3id.org/mlsea and https://w3id.org/mlsea-kg).
It links publication, repository and datset entities.

Input:
- Linked Papers With Code Dump in .nt format (accessible at https://zenodo.org/records/13881433)
- The file pwc_1.nt.gz from the MLSea Dump (accessible at https://zenodo.org/records/11264641)

Output:
- .../paper-sameas-links.nt --> The sameas links between the publication entities of LPWC and MLSea
- .../repository-sameas-links.nt --> The sameas links between the repository entities of LPWC and MLSea
- .../dataset-sameas-links.nt --> The sameas links between the dataset entities of LPWC and MLSea

For all code cells the file path needs to be adjusted to the local file path.
"""

In [ ]:
# Linking of Publications i.e. <https://linkedpaperswithcode.com/class/paper> (LPWC) and <http://w3id.org/mlso/ScientificWork> (MLSea) Entities.
# The linking is based on the entities title.

from rdflib import Graph, URIRef, Namespace
import re

# normalize publication titles for better matching
def normalize_title(title):
    return re.sub(r'\W+', '', title.lower()).strip()

mlsea_papers = {}
lpwc_papers = {}

# MLSea Papers
with open(".../pwc_1.nt", "r") as file:
    for line in file:
        if "<http://www.w3.org/2000/01/rdf-schema#label>" in line:
            parts = line.strip().split(" ", 2)
            uri = parts[0].strip("<>")
            title = re.search(r'"(.*?)"', parts[2]).group(1)
            normalized_title = normalize_title(title)
            mlsea_papers[normalized_title] = uri

# LPWC Papers
with open(".../linkedpaperswithcode_2024-09-05.nt", "r") as file:
    for line in file:
        if "<http://purl.org/dc/terms/title>" in line:
            parts = line.strip().split(" ", 2)
            uri = parts[0].strip("<>")
            title = re.search(r'"(.*?)"', parts[2]).group(1)
            normalized_title = normalize_title(title)
            lpwc_papers[normalized_title] = uri

# create RDF graph with owl:sameAs links
g = Graph()
OWL = Namespace("http://www.w3.org/2002/07/owl#")

for title, mlsea_uri in mlsea_papers.items():
    if title in lpwc_papers:
        lpwc_uri = lpwc_papers[title]
        g.add((URIRef(mlsea_uri), OWL.sameAs, URIRef(lpwc_uri)))

# postprocessing: only keep triples that have a MLSea scientific work as subject
filtered_graph = Graph()
prefix = "http://w3id.org/mlsea/pwc/scientificWork/"

for subj, pred, obj in g:
    if str(subj).startswith(prefix):
        filtered_graph.add((obj, pred, subj))

# save sameAs links to file
filtered_graph.serialize(destination=".../paper-sameas-links.nt", format="nt")

print(f"Mapping completed. Found {len(filtered_graph)} sameas links.")

In [ ]:
# Linking of Repositories i.e. <https://linkedpaperswithcode.com/class/repository> (LPWC) and <http://www.w3.org/ns/mls#Software> (MLSea) Entities.
# The linking is based on the LPWC entities URI and the MLSea entities codeRepository property.

from rdflib import Graph, URIRef, Namespace
import re

# normalize repository URLs
def normalize_url(url):
    url = re.sub(r'^https?://', '', url.lower())
    url = re.sub(r'^linkedpaperswithcode.com/repository/', '', url)
    return url.strip('/')

mlsea_repos = {}
lpwc_repos = {}

# MLSea
with open(".../pwc_1.nt", "r") as file:
    for line in file:
        if "<http://schema.org/codeRepository>" in line:
            parts = line.strip().split(" ", 2)
            repo_uri = parts[0].strip("<>")
            repo_url = parts[2].strip(" .").strip("<>")
            normalized_url = normalize_url(repo_url)
            mlsea_repos[normalized_url] = repo_uri

# LPWC
with open(".../linkedpaperswithcode_2024-09-05.nt", "r") as file:
    for line in file:
        if "<https://linkedpaperswithcode.com/class/repository>" in line:
            parts = line.strip().split(" ", 2)
            repo_uri = parts[0].strip("<>")
            normalized_url = normalize_url(repo_uri)
            lpwc_repos[normalized_url] = repo_uri

g = Graph()
OWL = Namespace("http://www.w3.org/2002/07/owl#")

# postprocessing: only keep triples that have a MLSea software as subject
for url, mlsea_uri in mlsea_repos.items():
    if url in lpwc_repos and mlsea_uri.startswith("http://w3id.org/mlsea/pwc/software/"):
        lpwc_uri = lpwc_repos[url]
        g.add((URIRef(lpwc_uri), OWL.sameAs, URIRef(mlsea_uri)))

g.serialize(destination=".../repository-sameas-links.nt", format="nt")

print(f"Mapping completed. Found {len(g)} sameas links.")


In [ ]:
# Linking of Datasets i.e. <https://linkedpaperswithcode.com/class/dataset> (LPWC) and <http://www.w3.org/ns/dcat#Dataset> (MLSea) Entities.
# The linking is based on the entities URIs.

from rdflib import Graph, URIRef, Namespace
import re
import urllib.parse

# normalize dataset URIs
def normalize_uri(uri):
    uri = uri.strip('<>')
    uri = urllib.parse.unquote(uri)
    uri = re.sub(r'^https?://(w3id.org/mlsea/pwc/dataset|linkedpaperswithcode.com/dataset)/', '', uri)
    return re.sub(r'\W+', '', uri.lower())

mlsea_datasets = {}
lpwc_datasets = {}

# MLSea
with open(".../pwc_1.nt", "r") as file:
    for line in file:
        if "<http://www.w3.org/ns/dcat#Dataset>" in line:
            parts = line.strip().split(" ", 2)
            uri = parts[0].strip("<>")
            normalized = normalize_uri(uri)
            mlsea_datasets[normalized] = uri

# LPWC
with open(".../linkedpaperswithcode_2024-09-05.nt", "r") as file:
    for line in file:
        if "<https://linkedpaperswithcode.com/class/dataset>" in line:
            parts = line.strip().split(" ", 2)
            uri = parts[0].strip("<>")
            normalized = normalize_uri(uri)
            lpwc_datasets[normalized] = uri

g = Graph()
OWL = Namespace("http://www.w3.org/2002/07/owl#")

for normalized, mlsea_uri in mlsea_datasets.items():
    if normalized in lpwc_datasets:
        lpwc_uri = lpwc_datasets[normalized]
        g.add((URIRef(lpwc_uri), OWL.sameAs, URIRef(mlsea_uri)))
    
g.serialize(destination=".../dataset-sameas-links.nt", format="nt")

print(f"Mapping completed. Found {len(g)} sameas links.")
